In [1]:
from MotoricaInterface import MotoricaSerialInterface

import numpy as np
import time
import pandas as pd

In [2]:
from serial.tools import list_ports
pd.DataFrame([c.__dict__ for c in list_ports.comports()])

,device,name,description,hwid,vid,pid,serial_number,location,manufacturer,product,interface
0,/dev/cu.BLTH,cu.BLTH,n/a,n/a,NaN,NaN,None,None,None,None,None
1,/dev/cu.MotoricasMacBookPro,cu.MotoricasMacBookPro,n/a,n/a,NaN,NaN,None,None,None,None,None
2,/dev/cu.ESP32-BT-Slave,cu.ESP32-BT-Slave,n/a,n/a,NaN,NaN,None,None,None,None,None
3,/dev/cu.BoseQC35II,cu.BoseQC35II,n/a,n/a,NaN,NaN,None,None,None,None,None
4,/dev/cu.ESP32Bluetooth,cu.ESP32Bluetooth,n/a,n/a,NaN,NaN,None,None,None,None,None
5,/dev/cu.Bluetooth-Incoming-Port,cu.Bluetooth-Incoming-Port,n/a,n/a,NaN,NaN,None,None,None,None,None
6,/dev/cu.usbserial-1440,cu.usbserial-1440,USB Serial,USB VID:PID=1A86:7523 LOCATION=20-4,6790.0,29987.0,None,20-4,None,USB Serial,None


In [3]:
ser = MotoricaSerialInterface('/dev/cu.usbserial-1440', debug=False, logging_enabled=False)
ser.serial

Serial<id=0x114ca9f60, open=True>(port='/dev/cu.usbserial-1440', baudrate=115200, bytesize=8, parity='N', stopbits=1, timeout=0.05, xonxoff=False, rtscts=False, dsrdtr=False)

In [4]:
ser.start()

Start Serial Reading


# Simple Pack Counter

In [ ]:
ser.DEBUG = False
pack_counter = 0;
t_start = time.time()
t_last_pack = 0;

ser.flush_rx_packs()

while True:
    if (len(ser.rx_pack)):
        now = time.time()
        ts, RX = ser.get_data(1)[0]
        print(f'N={pack_counter}\tFrom thread {ts-ser.thread_start_time:.3f}\tFrom main {time.time() - t_start:.2f}\tPACK_HEADER {RX[:2]}\tTS_DIFF={(now - t_last_pack)*1000:.0f}', 
              end = ' '*20 + '\r')
        t_last_pack = now
        pack_counter+=1
        
    time.sleep(0.001) 

# EMG Pack reader

In [5]:
DEBUG = False
pack_counter = 0;
t_start = time.time()

while True:
    
    if (NEWPACK):
        NEWPACK = False
        pack_counter += 1
        if (RX[0] == 0x01) & (RX[1] == 0x20):
            print(f'{pack_counter}\t{ time.time() - t_start:.2f}\tEMG PACK {RX[3]:03d} {RX[4]:03d}', end = ' '*20 + '\r')
        
    time.sleep(0.001) 

KeyboardInterrupt: 

# Start training

In [6]:
import socket

ip = 'localhost'
pronation = 0.5

def socket_send(cmd):
#     return None
    s = socket.socket()          
    s.connect((ip, 1755))
    s.send(cmd.encode())
    s.close()

In [ ]:
DEBUG = False
pack_counter = 0;
t_start = time.time()

pack = (np.array([0x01, 0x82, 0x00, 0x00], dtype='uint8')) # protocol
pack[-1] = get_crc(pack, len(pack)-1)

ser.write(pack)
time.sleep(0.2) # wait till something happens

socket_send(f'0,0,0,0,0,0,0,0,0,0,0,0,{pronation}') # reset v-hand
while True:
    if (NEWPACK):
        NEWPACK = False
        pack_counter += 1
        
        if (RX[0] == 0x01) & (RX[1] == 0x20):
            print(f'{pack_counter}\t{ time.time() - t_start:.2f}\tEMG PACK {RX[3]:03d} {RX[4]:03d}', end = ' '*20 + '\r')
            if ((RX[3] == 0) & (RX[4] == 0)):
                socket_send(f'0,0,0,0,0,0,0,0,0,0,0,0,{pronation}') # reset v-hand
            elif RX[3] > RX[4]:
                socket_send(f'0,0,0,0,0,0,0,0,0,0,1,0,{pronation}')
            else:
                socket_send(f'0,0,0,0,0,0,0,0,0,0,0,1,{pronation}')
            
        if (RX[0] == 0x01) & (RX[1] == 0x13):
            print(f'{pack_counter}\t{ time.time() - t_start:.2f}\tPACK STOP', end = ' '*20 + '\r')
            socket_send(f'0,0,0,0,0,0,0,0,0,0,0,0,{pronation}')
            
        if (RX[0] == 0x01) & (RX[1] == 0x10):
            print(f'{pack_counter}\t{ time.time() - t_start:.2f}\tPACK CLOSE', end = ' '*20 + '\r')
            socket_send(f'0,0,0,0,0,0,0,0,0,0,1,0,{pronation}')
            
        if (RX[0] == 0x01) & (RX[1] == 0x0F):
            print(f'{pack_counter}\t{ time.time() - t_start:.2f}\tPACK OPEN', end = ' '*20 + '\r')
            socket_send(f'0,0,0,0,0,0,0,0,0,0,0,1,{pronation}')
        
    time.sleep(0.001) 

Exception in thread Thread-5:
Traceback (most recent call last):
  File "/home/user/miniconda3/envs/tf214/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/home/user/miniconda3/envs/tf214/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_43860/4087773592.py", line 26, in serial_parser
  File "/home/user/miniconda3/envs/tf214/lib/python3.9/site-packages/serial/serialposix.py", line 595, in read
    raise SerialException(
serial.serialutil.SerialException: device reports readiness to read but returned no data (device disconnected or multiple access on port?)


In [11]:
ser.write(pack)
time.sleep(0.2) # wait till something happens